# 01 — Collecter PokéAPI par CSV et par API

**Objectif :** obtenir deux tables au même format, une ligne par forme, puis vérifier qu’elles contiennent les mêmes observations et les mêmes valeurs avant une future fusion avec Showdown. Aucune forme n’est filtrée ou regroupée, même si ses statistiques sont identiques à celles d’une autre forme. Talents et conditions d’évolution sont hors périmètre.

Les fonctions sont dans `src/collect.py`, les adresses et instantanés dans `config/sources.json`. Exécuter les cinq cellules de code dans l’ordre pour vérifier les deux méthodes. La collecte CSV et la collecte API sont deux cellules séparées : après l’import, on peut aussi lancer uniquement celle dont on a besoin.

**Deux versions différentes peuvent diverger :** les CSV sont figés à une révision du dépôt, tandis que l’API publique est actuelle au moment du téléchargement. Le code vérifie l’égalité ; il ne modifie pas une source pour la forcer.

In [1]:
from pathlib import Path
import sys
from IPython.display import display

project_dir = Path.cwd()
if project_dir.name == "notebook":
    project_dir = project_dir.parent
assert (project_dir / "src" / "collect.py").exists(), "Ouvrir depuis le projet ou notebook/."
if str(project_dir) not in sys.path:
    sys.path.insert(0, str(project_dir))

from src.collect import (
    collect_sources, load_sources, check_sources, pokeapi_from_csv,
    collect_pokeapi_api, compare_pokeapi, export_collections,
)

C:\Users\Ariel\AppData\Local\Programs\Python\Python311\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.7.0) or chardet (7.4.3)/charset_normalizer (3.4.7) doesn't match a supported version!
  warnings.warn(


## 1. Méthode CSV : télécharger, contrôler, mettre au format commun

Les fichiers bruts restent séparés et inchangés. `collect_sources()` vérifie leurs empreintes et réutilise l’instantané présent. Cette étape conserve aussi Showdown, qui sera utilisé plus tard : **il n’entre pas dans la table PokéAPI comparée**.

`pokeapi_from_csv()` rattache à chaque forme sa variété, son espèce, ses six statistiques, son groupe de jeux et ses noms. Les types explicites de forme ont priorité ; à défaut, on utilise ceux de la variété. Il s’agit d’une mise au même format que l’API, pas d’une correction des sources.

In [2]:
data_dir = collect_sources()
data = load_sources()
source_checks = check_sources(data)
pokeapi_csv = pokeapi_from_csv(data)
print("Table CSV :", pokeapi_csv.shape)
display(pokeapi_csv.head())

Conservé et vérifié : pokemon_species.csv
Conservé et vérifié : pokemon.csv
Conservé et vérifié : pokemon_stats.csv
Conservé et vérifié : stats.csv
Conservé et vérifié : pokemon_species_names.csv
Conservé et vérifié : pokemon_types.csv
Conservé et vérifié : type_names.csv
Conservé et vérifié : showdown_pokedex.json
Conservé et vérifié : pokemon_forms.csv
Conservé et vérifié : pokemon_form_types.csv
Conservé et vérifié : version_groups.csv
Contrôles terminés : 1579 formes conservées ; aucun filtre ni regroupement.


Table CSV : (1579, 27)


,form_id,pokemon_id,species_id,forme,variete,espece,nom_fr,genre_en,version_group_id,generation,...,hp,attack,defense,special-attack,special-defense,speed,type_1_id,type_2_id,type_1,type_2
0,1,1,1,bulbasaur,bulbasaur,bulbasaur,Bulbizarre,Seed Pokémon,28,1,...,45,49,49,65,65,45,12,4,Plante,Poison
1,2,2,2,ivysaur,ivysaur,ivysaur,Herbizarre,Seed Pokémon,28,1,...,60,62,63,80,80,60,12,4,Plante,Poison
2,3,3,3,venusaur,venusaur,venusaur,Florizarre,Seed Pokémon,28,1,...,80,82,83,100,100,80,12,4,Plante,Poison
3,4,4,4,charmander,charmander,charmander,Salamèche,Lizard Pokémon,28,1,...,39,52,43,60,50,65,10,<NA>,Feu,<NA>
4,5,5,5,charmeleon,charmeleon,charmeleon,Reptincel,Flame Pokémon,28,1,...,58,64,58,80,65,80,10,<NA>,Feu,<NA>


## 2. Méthode API : parcourir toutes les formes et leurs références

On parcourt **toutes les pages du catalogue `/pokemon-form/`**, indépendamment de la liste des CSV. Chaque forme fournit l’adresse de sa variété et de son groupe de jeux ; la variété fournit l’adresse de son espèce et les statistiques. Ainsi, aucune génération n’est fixée à la main et la génération de l’espèce ne remplace pas celle de la forme.

Les JSON sont sauvegardés dans `data/raw/api_01/` avec adresse, date et empreinte. La première collecte peut durer plusieurs minutes ; les suivantes réutilisent le cache. Une erreur HTTP arrête la collecte, sans repli silencieux. Après une interruption, relancer cette cellule reprend les téléchargements manquants. L’API peut évoluer durant une collecte : les dates de chaque réponse sont conservées.

Les types viennent de la forme API, y compris pour Arceus et Silvallié. Les statistiques viennent directement de sa variété : aucun appel `/pokemon/nom-de-forme` n’est tenté.

In [3]:
pokeapi_api = collect_pokeapi_api()
print("Table API :", pokeapi_api.shape)
display(pokeapi_api.head())

Catalogue API : 1579 formes. Les réponses en cache seront réutilisées.


Formes lues : 100/1579


Formes lues : 200/1579


Formes lues : 300/1579


Formes lues : 400/1579


Formes lues : 500/1579


... (9 outputs left out)

Formes lues : 1500/1579


Formes lues : 1579/1579


Table API : (1579, 27)


,form_id,pokemon_id,species_id,forme,variete,espece,nom_fr,genre_en,version_group_id,generation,...,hp,attack,defense,special-attack,special-defense,speed,type_1_id,type_2_id,type_1,type_2
0,1,1,1,bulbasaur,bulbasaur,bulbasaur,Bulbizarre,Seed Pokémon,28,1,...,45,49,49,65,65,45,12,4,Plante,Poison
1,2,2,2,ivysaur,ivysaur,ivysaur,Herbizarre,Seed Pokémon,28,1,...,60,62,63,80,80,60,12,4,Plante,Poison
2,3,3,3,venusaur,venusaur,venusaur,Florizarre,Seed Pokémon,28,1,...,80,82,83,100,100,80,12,4,Plante,Poison
3,4,4,4,charmander,charmander,charmander,Salamèche,Lizard Pokémon,28,1,...,39,52,43,60,50,65,10,<NA>,Feu,<NA>
4,5,5,5,charmeleon,charmeleon,charmeleon,Reptincel,Flame Pokémon,28,1,...,58,64,58,80,65,80,10,<NA>,Feu,<NA>


## 3. Comparer les deux résultats, sans Showdown

La table commune contient **27 colonnes** :

- `form_id` (clé), `pokemon_id`, `species_id` et les noms techniques `forme`, `variete`, `espece` ;
- `nom_fr` : nom français de l’espèce, et `genre_en` : catégorie anglaise utile pour les Paradoxes ;
- `version_group_id`, `generation` (introduction de la forme) et `generation_espece`, conservée séparément ;
- `is_default_form`, `is_default_variety`, `is_battle_only`, `is_mega`, `is_legendary`, `is_mythical` ;
- `hp`, `attack`, `defense`, `special-attack`, `special-defense`, `speed` ;
- `type_1_id`, `type_2_id`, `type_1`, `type_2` (libellés français).

L’ordre des lignes et les différences de représentation Python (entiers/booléens, valeurs manquantes) sont harmonisés. Les noms, nombres et valeurs ne sont pas corrigés. Deux seconds types absents sont égaux ; une absence d’un seul côté est un écart. Les formes présentes dans une seule méthode sont également signalées : on ne compare pas seulement leur intersection.

**Portée du résultat :** l’égalité concerne ces 27 colonnes utiles au projet, pas tous les champs disponibles dans les fichiers bruts. Les habitats, ordres d’affichage, autres traductions, etc. restent accessibles dans les sources mais ne sont pas déclarés identiques par ce contrôle.

In [4]:
comparaison = compare_pokeapi(pokeapi_csv, pokeapi_api)
display(comparaison["resume"])
display(comparaison["ecarts"].head(20))

# Les écarts complets sont sauvegardés, même si le contrôle final échoue.
output_dir = export_collections(pokeapi_csv, pokeapi_api, comparaison)
print("Tables et rapport enregistrés dans :", output_dir)

,nombre
Formes CSV,1579
Formes API,1579
Formes communes,1579
Formes seulement CSV,0
Formes seulement API,0
Colonnes comparées hors identifiant,26
Cellules différentes sur les formes communes,0


,form_id,champ,csv,api


Tables et rapport enregistrés dans : C:\Users\Ariel\OneDrive - ENSEA\Documents\Python pour la DS\Projet\Projet-python-data-science\data\collected\reference_01_vs_api_01


## 4. Valider avant de poursuivre

Si le contrôle échoue, consulter `comparaison["ecarts"]` ou `comparaison_ecarts.csv` : les identifiants, champs et valeurs des deux sources y figurent. Vérifier ensuite les réponses brutes et leurs dates. **Aucune fusion automatique des deux collectes, aucun remplacement des CSV et aucune suppression de forme ne sont effectués.**

Si le contrôle réussit, les deux tables peuvent servir de façon équivalente pour les variables comparées. On choisira **une** table pour la future fusion avec Showdown ; les concaténer doublerait les observations. Les notebooks EDA et préparation existants continuent à lire les tables CSV brutes ; ils ne basculent pas automatiquement sur l’API.

Les exports sont dans `data/collected/reference_01_vs_api_01/`. La future collecte volontairement actualisée doit utiliser un nouvel identifiant d’instantané API ; ne pas effacer quelques réponses d’un cache complet pour les remplacer par des données plus récentes.

**Résultat de l’exécution vérifiée :** 1 579 formes dans chaque méthode, 27 colonnes au même format, aucune forme absente et aucune cellule différente. Ce constat vaut pour `reference_01` et le cache `api_01` ; une actualisation doit être contrôlée à nouveau.

In [5]:
assert comparaison["identiques"], "Écarts CSV/API : examiner le rapport avant de choisir une source."
print(f"Égalité vérifiée : {len(pokeapi_csv)} formes et {len(pokeapi_csv.columns)} colonnes, hors Showdown.")

Égalité vérifiée : 1579 formes et 27 colonnes, hors Showdown.
